# Notebook 08 — Stage K Representation & Unsupervised Layer

Purpose:
- map patient-state structure in a lower-dimensional space
- detect latent cohorts via density-aware clustering
- connect discovered clusters to escalation behavior

## Planned Tasks (Notebook 08)

- [ ] Load Stage J selected model context and canonical panel
- [ ] Build standardized representation matrix for Stage K
- [ ] Generate PCA and optional UMAP/TSNE representation plots
- [ ] Run HDBSCAN (fallback DBSCAN) clustering and profile cohorts
- [ ] Quantify cluster stability and link clusters to escalation outcomes
- [ ] Export Stage K report + manifest + checklist proof

In [1]:
from pathlib import Path
import os
import gc
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
TABLE_DIR = PROJECT_ROOT / 'Results' / 'tables' / 'notebook08_stage_k'
FIG_DIR = PROJECT_ROOT / 'Results' / 'figures' / 'notebook08_stage_k'
REPORT_DIR = PROJECT_ROOT / 'Results' / 'reports' / 'notebook08_stage_k'
META_DIR = PROJECT_ROOT / 'Data' / 'metadata'
for d in [TABLE_DIR, FIG_DIR, REPORT_DIR, META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

STAGE_K_MAX_ROWS = int(os.getenv('STAGE_K_MAX_ROWS', '80000'))
STAGE_K_BOOTSTRAP_RUNS = int(os.getenv('STAGE_K_BOOTSTRAP_RUNS', '10'))

print('Notebook 08 workspace ready')
print('Crash-prevention caps -> STAGE_K_MAX_ROWS:', STAGE_K_MAX_ROWS, '| STAGE_K_BOOTSTRAP_RUNS:', STAGE_K_BOOTSTRAP_RUNS)

Notebook 08 workspace ready
Crash-prevention caps -> STAGE_K_MAX_ROWS: 80000 | STAGE_K_BOOTSTRAP_RUNS: 10


In [2]:
# Stage J handoff + canonical matrix build
phase_j_manifest_path = META_DIR / 'phase_j_manifest.json'
phase_j_proof_path = PROJECT_ROOT / 'Results' / 'reports' / 'notebook07_stage_j' / 'stage_j_checklist_proof.json'
if not phase_j_manifest_path.exists():
    raise FileNotFoundError(f'Missing Stage J manifest: {phase_j_manifest_path}')
if not phase_j_proof_path.exists():
    raise FileNotFoundError(f'Missing Stage J proof: {phase_j_proof_path}')

with open(phase_j_manifest_path, 'r', encoding='utf-8') as f:
    phase_j_manifest = json.load(f)
with open(phase_j_proof_path, 'r', encoding='utf-8') as f:
    phase_j_proof = json.load(f)

selected_j = phase_j_manifest.get('selected_model', phase_j_proof.get('selected', {}).get('model', 'unknown'))
print('Stage J selected model for Stage K context:', selected_j)

np.random.seed(42)
panel_path = PROJECT_ROOT / 'Results' / 'tables' / 'notebook03_phase_f' / 'phase_f_closed_loop_panel.parquet'
cols = [
    'patient_id', 'day', 'I_stage_f_base', 'hazard_prob_stage_f', 'stage_f_escalation_event',
    'cycle_id_stage_f', 'months_since_cycle_start', 'response_state_active'
 ]
data = pd.read_parquet(panel_path, columns=cols).copy()

p_missing_instability = np.clip(0.03 + 0.12 * (data['response_state_active'].eq('nonresponse')).astype(float), 0.0, 0.35)
p_missing_hazard = np.clip(0.02 + 0.10 * (data['months_since_cycle_start'] > 18).astype(float), 0.0, 0.25)
p_missing_cycle = np.clip(0.01 + 0.08 * (data['stage_f_escalation_event'] == 0).astype(float), 0.0, 0.20)

u1 = np.random.rand(len(data))
u2 = np.random.rand(len(data))
u3 = np.random.rand(len(data))
data.loc[u1 < p_missing_instability, 'I_stage_f_base'] = np.nan
data.loc[u2 < p_missing_hazard, 'hazard_prob_stage_f'] = np.nan
data.loc[u3 < p_missing_cycle, 'months_since_cycle_start'] = np.nan

for c in ['I_stage_f_base', 'hazard_prob_stage_f', 'months_since_cycle_start']:
    data[f'{c}__is_missing'] = data[c].isna().astype(np.int8)

data['is_nonresponse'] = data['response_state_active'].eq('nonresponse').astype(np.int8)
data['is_partial'] = data['response_state_active'].eq('partial_response').astype(np.int8)
data['is_stabilized'] = data['response_state_active'].eq('stabilized').astype(np.int8)
data['I_stage_f_base_x_cycle'] = data['I_stage_f_base'].fillna(data['I_stage_f_base'].median()) * data['cycle_id_stage_f']
data['extreme_hazard_flag'] = (
    data['hazard_prob_stage_f'].fillna(data['hazard_prob_stage_f'].median())
    >= data['hazard_prob_stage_f'].fillna(data['hazard_prob_stage_f'].median()).quantile(0.995)
).astype(np.int8)

feature_set = [
    'cycle_id_stage_f', 'months_since_cycle_start', 'I_stage_f_base', 'hazard_prob_stage_f',
    'is_nonresponse', 'is_partial', 'is_stabilized',
    'I_stage_f_base__is_missing', 'hazard_prob_stage_f__is_missing', 'months_since_cycle_start__is_missing',
    'I_stage_f_base_x_cycle', 'extreme_hazard_flag'
]

sample_n = min(STAGE_K_MAX_ROWS, len(data))
k_df = data.sample(n=sample_n, random_state=42).reset_index(drop=True)
X_raw = k_df[feature_set].copy()
y = k_df['stage_f_escalation_event'].astype(int).to_numpy()

imp = SimpleImputer(strategy='median')
scaler = StandardScaler()
X_imp = imp.fit_transform(X_raw).astype(np.float32, copy=False)
X_scaled = scaler.fit_transform(X_imp).astype(np.float32, copy=False)
del X_imp, X_raw
gc.collect()

print('Stage K canonical matrix shape:', X_scaled.shape)
print('Escalation prevalence in sample:', float(y.mean()))

Stage J selected model for Stage K context: hist_gb_strong
Stage K canonical matrix shape: (80000, 12)
Escalation prevalence in sample: 0.0116875


In [3]:
# Representation maps: PCA + optional UMAP (fallback TSNE)
pca = PCA(n_components=2, random_state=42)
pca_2d = pca.fit_transform(X_scaled)

rep_df = pd.DataFrame({
    'pca_1': pca_2d[:, 0],
    'pca_2': pca_2d[:, 1],
    'escalation': y,
    'response_state_active': k_df['response_state_active'].astype(str).to_numpy()
})

umap_used = False
try:
    import umap
    reducer = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.08, random_state=42)
    umap_2d = reducer.fit_transform(X_scaled)
    rep_df['embed_1'] = umap_2d[:, 0]
    rep_df['embed_2'] = umap_2d[:, 1]
    rep_label = 'UMAP'
    umap_used = True
except Exception:
    tsne_n = min(len(X_scaled), 7000)
    tsne_idx = np.random.RandomState(42).choice(len(X_scaled), size=tsne_n, replace=False)
    tsne = TSNE(n_components=2, perplexity=35, learning_rate='auto', init='pca', random_state=42)
    tsne_2d = tsne.fit_transform(X_scaled[tsne_idx])
    rep_df.loc[tsne_idx, 'embed_1'] = tsne_2d[:, 0]
    rep_df.loc[tsne_idx, 'embed_2'] = tsne_2d[:, 1]
    rep_label = 'TSNE_fallback'

rep_df.to_csv(TABLE_DIR / 'stage_k_representation_coordinates.csv', index=False)

plt.figure(figsize=(8, 6))
plt.scatter(rep_df['pca_1'], rep_df['pca_2'], c=rep_df['escalation'], s=3, alpha=0.25)
plt.title('Stage K PCA Representation (colored by escalation)')
plt.xlabel('PCA-1')
plt.ylabel('PCA-2')
plt.tight_layout()
plt.savefig(FIG_DIR / 'stage_k_pca_representation.png', dpi=140, bbox_inches='tight')
plt.close()

embed_non_null = rep_df[['embed_1', 'embed_2']].dropna()
if len(embed_non_null) > 0:
    plt.figure(figsize=(8, 6))
    idx = embed_non_null.index
    plt.scatter(rep_df.loc[idx, 'embed_1'], rep_df.loc[idx, 'embed_2'], c=rep_df.loc[idx, 'escalation'], s=3, alpha=0.25)
    plt.title(f'Stage K {rep_label} Representation (colored by escalation)')
    plt.xlabel('Embed-1')
    plt.ylabel('Embed-2')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'stage_k_embed_representation.png', dpi=140, bbox_inches='tight')
    plt.close()

with open(REPORT_DIR / 'stage_k_representation_summary.txt', 'w', encoding='utf-8') as f:
    f.write('Stage K Representation Summary\n')
    f.write(f'pca_variance_explained: {pca.explained_variance_ratio_.tolist()}\n')
    f.write(f'embed_method: {rep_label}\n')
    f.write(f'umap_used: {umap_used}\n')
    f.write(f'n_rows: {len(rep_df)}\n')

print('Stage K representation artifacts generated')
print('-', (TABLE_DIR / 'stage_k_representation_coordinates.csv').exists())
print('-', (FIG_DIR / 'stage_k_pca_representation.png').exists())
print('-', (FIG_DIR / 'stage_k_embed_representation.png').exists())
print('-', (REPORT_DIR / 'stage_k_representation_summary.txt').exists())
rep_df.head(5)

Stage K representation artifacts generated
- True
- True
- True
- True


,pca_1,pca_2,escalation,response_state_active,embed_1,embed_2
0,2.223393,0.322437,0,stabilized,NaN,NaN
1,-0.780365,1.202280,0,pre_admission,NaN,NaN
2,-0.693131,-0.453013,0,pre_admission,NaN,NaN
3,2.608114,1.234326,0,stabilized,NaN,NaN
4,-0.977479,0.284055,0,pre_admission,NaN,NaN


In [4]:
# Clustering + profile + stability diagnostics
cluster_input = pca_2d.astype(np.float32, copy=False)
cluster_method = 'hdbscan'

try:
    import hdbscan
    clusterer = hdbscan.HDBSCAN(min_cluster_size=280, min_samples=25)
    cluster_labels = clusterer.fit_predict(cluster_input)
except Exception:
    cluster_method = 'dbscan_fallback'
    clusterer = DBSCAN(eps=0.42, min_samples=70)
    cluster_labels = clusterer.fit_predict(cluster_input)

k_df['cluster'] = cluster_labels
rep_df['cluster'] = cluster_labels

valid_mask = cluster_labels >= 0
if valid_mask.sum() > 100 and len(np.unique(cluster_labels[valid_mask])) > 1:
    sil = float(silhouette_score(cluster_input[valid_mask], cluster_labels[valid_mask]))
else:
    sil = np.nan

profile = k_df.groupby('cluster').agg(
    n=('cluster', 'size'),
    escalation_rate=('stage_f_escalation_event', 'mean'),
    mean_cycle=('cycle_id_stage_f', 'mean'),
    mean_hazard=('hazard_prob_stage_f', 'mean'),
    mean_instability=('I_stage_f_base', 'mean')
).reset_index().sort_values('n', ascending=False)
profile.to_csv(TABLE_DIR / 'stage_k_cluster_profile.csv', index=False)

state_mix = pd.crosstab(k_df['cluster'], k_df['response_state_active'], normalize='index').reset_index()
state_mix.to_csv(TABLE_DIR / 'stage_k_cluster_state_mix.csv', index=False)

plt.figure(figsize=(8, 6))
plt.scatter(rep_df['pca_1'], rep_df['pca_2'], c=rep_df['cluster'], s=3, alpha=0.35, cmap='tab20')
plt.title(f'Stage K Clusters on PCA ({cluster_method})')
plt.xlabel('PCA-1')
plt.ylabel('PCA-2')
plt.tight_layout()
plt.savefig(FIG_DIR / 'stage_k_cluster_map.png', dpi=140, bbox_inches='tight')
plt.close()

# simple stability: rerun clustering on bootstrap PCA samples and compare ARI on overlap
rng = np.random.default_rng(42)
stability_scores = []
base_idx = np.arange(len(cluster_labels))
for _ in range(STAGE_K_BOOTSTRAP_RUNS):
    sample_idx = rng.choice(base_idx, size=int(0.75 * len(base_idx)), replace=False)
    sample_points = cluster_input[sample_idx]
    try:
        if cluster_method == 'hdbscan':
            import hdbscan
            sub_clusterer = hdbscan.HDBSCAN(min_cluster_size=220, min_samples=20)
            sub_labels = sub_clusterer.fit_predict(sample_points)
        else:
            sub_clusterer = DBSCAN(eps=0.42, min_samples=60)
            sub_labels = sub_clusterer.fit_predict(sample_points)

        base_labels_sub = cluster_labels[sample_idx]
        valid = (base_labels_sub >= 0) & (sub_labels >= 0)
        if valid.sum() > 50 and len(np.unique(base_labels_sub[valid])) > 1 and len(np.unique(sub_labels[valid])) > 1:
            stability_scores.append(float(adjusted_rand_score(base_labels_sub[valid], sub_labels[valid])))
    except Exception:
        continue

stability_df = pd.DataFrame({
    'cluster_method': [cluster_method],
    'n_clusters_excluding_noise': [int(len(set(cluster_labels[cluster_labels >= 0])))],
    'noise_fraction': [float((cluster_labels < 0).mean())],
    'silhouette_valid': [None if pd.isna(sil) else float(sil)],
    'bootstrap_ari_mean': [float(np.mean(stability_scores)) if len(stability_scores) > 0 else np.nan],
    'bootstrap_ari_std': [float(np.std(stability_scores)) if len(stability_scores) > 0 else np.nan],
    'bootstrap_runs_used': [int(len(stability_scores))]
})
stability_df.to_csv(TABLE_DIR / 'stage_k_cluster_stability.csv', index=False)

with open(REPORT_DIR / 'stage_k_cluster_interpretation_summary.txt', 'w', encoding='utf-8') as f:
    f.write('Stage K Cluster Interpretation Summary\n')
    f.write(f'cluster_method: {cluster_method}\n')
    f.write(f'n_clusters_excluding_noise: {int(len(set(cluster_labels[cluster_labels >= 0])))}\n')
    f.write(f'noise_fraction: {float((cluster_labels < 0).mean()):.6f}\n')
    f.write(f'silhouette_valid: {sil}\n')
    f.write(f'bootstrap_ari_mean: {stability_df["bootstrap_ari_mean"].iloc[0]}\n')
    f.write(f'bootstrap_runs_used: {int(stability_df["bootstrap_runs_used"].iloc[0])}\n')

print('Stage K clustering artifacts generated')
print('-', (TABLE_DIR / 'stage_k_cluster_profile.csv').exists())
print('-', (TABLE_DIR / 'stage_k_cluster_state_mix.csv').exists())
print('-', (TABLE_DIR / 'stage_k_cluster_stability.csv').exists())
print('-', (FIG_DIR / 'stage_k_cluster_map.png').exists())
print('-', (REPORT_DIR / 'stage_k_cluster_interpretation_summary.txt').exists())
profile.head(10)

Stage K clustering artifacts generated
- True
- True
- True
- True
- True


,cluster,n,escalation_rate,mean_cycle,mean_hazard,mean_instability
2,1,60319,0.000000,0.000000,0.006920,0.231563
1,0,18892,0.047216,1.123915,0.006532,0.272834
0,-1,479,0.089770,2.298539,0.007931,0.448686
3,2,310,0.000000,0.000000,0.013081,0.644934


In [5]:
# Stage K manifest + checklist proof
manifest_k = {
    'phase': 'K',
    'notebook': '08_stage_k_representation_unsupervised.ipynb',
    'inputs': [
        'Data/metadata/phase_j_manifest.json',
        'Results/reports/notebook07_stage_j/stage_j_checklist_proof.json',
        'Results/tables/notebook03_phase_f/phase_f_closed_loop_panel.parquet'
    ],
    'outputs_tables': [
        'Results/tables/notebook08_stage_k/stage_k_representation_coordinates.csv',
        'Results/tables/notebook08_stage_k/stage_k_cluster_profile.csv',
        'Results/tables/notebook08_stage_k/stage_k_cluster_state_mix.csv',
        'Results/tables/notebook08_stage_k/stage_k_cluster_stability.csv'
    ],
    'outputs_figures': [
        'Results/figures/notebook08_stage_k/stage_k_pca_representation.png',
        'Results/figures/notebook08_stage_k/stage_k_embed_representation.png',
        'Results/figures/notebook08_stage_k/stage_k_cluster_map.png'
    ],
    'outputs_reports': [
        'Results/reports/notebook08_stage_k/stage_k_representation_summary.txt',
        'Results/reports/notebook08_stage_k/stage_k_cluster_interpretation_summary.txt'
    ]
}
with open(META_DIR / 'phase_k_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest_k, f, indent=4)

proof_k = {
    'representation_coords_generated': (TABLE_DIR / 'stage_k_representation_coordinates.csv').exists(),
    'cluster_profile_generated': (TABLE_DIR / 'stage_k_cluster_profile.csv').exists(),
    'cluster_stability_generated': (TABLE_DIR / 'stage_k_cluster_stability.csv').exists(),
    'representation_figures_generated': (FIG_DIR / 'stage_k_pca_representation.png').exists() and (FIG_DIR / 'stage_k_cluster_map.png').exists(),
    'reports_generated': (REPORT_DIR / 'stage_k_representation_summary.txt').exists() and (REPORT_DIR / 'stage_k_cluster_interpretation_summary.txt').exists(),
    'manifest_generated': (META_DIR / 'phase_k_manifest.json').exists()
}
with open(REPORT_DIR / 'stage_k_checklist_proof.json', 'w', encoding='utf-8') as f:
    json.dump({'proof': proof_k, 'cluster_method': cluster_method}, f, indent=4)

print('Stage K manifest/proof generated')
for k, v in proof_k.items():
    print('-', k, ':', v)
proof_k

Stage K manifest/proof generated
- representation_coords_generated : True
- cluster_profile_generated : True
- cluster_stability_generated : True
- representation_figures_generated : True
- reports_generated : True
- manifest_generated : True


{'representation_coords_generated': True,
 'cluster_profile_generated': True,
 'cluster_stability_generated': True,
 'representation_figures_generated': True,
 'reports_generated': True,
 'manifest_generated': True}

In [ ]:
# Inline artifact gallery for this notebook stage
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image, Markdown

ROOT = PROJECT_ROOT if 'PROJECT_ROOT' in globals() else (Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd())
STAGE_PREFIX = 'notebook08'

def _match_stage_dirs(base, prefix):
    if not base.exists():
        return []
    return sorted([p for p in base.glob(f'{prefix}*') if p.is_dir()])

def _show_table_file(path):
    suffix = path.suffix.lower()
    display(Markdown(f'**{path.name}**'))
    try:
        if suffix == '.csv':
            display(pd.read_csv(path).head(200))
        elif suffix == '.parquet':
            display(pd.read_parquet(path).head(200))
        elif suffix == '.json':
            data = json.loads(path.read_text(encoding='utf-8'))
            if isinstance(data, list):
                display(pd.DataFrame(data).head(200))
            elif isinstance(data, dict):
                display(pd.DataFrame([data]).T.head(200))
            else:
                print(str(data)[:12000])
        elif suffix in {'.txt', '.md'}:
            print(path.read_text(encoding='utf-8')[:12000])
    except Exception as exc:
        print(f'Could not render {path.name}: {exc}')

table_dirs = _match_stage_dirs(ROOT / 'Results' / 'tables', STAGE_PREFIX)
figure_dirs = _match_stage_dirs(ROOT / 'Results' / 'figures', STAGE_PREFIX)
report_dirs = _match_stage_dirs(ROOT / 'Results' / 'reports', STAGE_PREFIX)

display(Markdown(f'## Inline Artifact Gallery: {STAGE_PREFIX}'))
if not table_dirs and not figure_dirs and not report_dirs:
    print('No stage-matched artifact folders found yet. Run generation cells first.')

for d in table_dirs:
    display(Markdown(f'### Tables ({d.name})'))
    files = sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in {'.csv', '.parquet', '.json', '.txt'}])
    if not files:
        print('No table files found')
    for fp in files:
        _show_table_file(fp)

for d in figure_dirs:
    display(Markdown(f'### Visualizations ({d.name})'))
    imgs = sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in {'.png', '.jpg', '.jpeg'}])
    if not imgs:
        print('No figure files found')
    for fp in imgs:
        display(Markdown(f'**{fp.name}**'))
        display(Image(filename=str(fp)))

for d in report_dirs:
    display(Markdown(f'### Reports ({d.name})'))
    files = sorted([p for p in d.iterdir() if p.is_file() and p.suffix.lower() in {'.csv', '.json', '.txt', '.md'}])
    if not files:
        print('No report files found')
    for fp in files:
        _show_table_file(fp)